# NHANES 2013–2014: High Protein Intake × Physical Activity × Kidney Function

Colab/GitHub용 정리본입니다. 노트북 하나로 데이터 다운로드, 전처리, 지표 생성, 시각화, 회귀분석, 결과 저장까지 실행할 수 있도록 구성했습니다.

## 핵심 수정사항
- 파일 탐색 로직 수정: questionnaire.csv / examination.csv / labs.csv를 명시적으로 인식
- 잘못된 변수 매핑 수정: weight_kg는 BMXWT, BMI는 BMXBMI, BUN은 LBXSBU
- DR1TACAR(알파카로틴)를 'animal protein'으로 잘못 사용하던 부분 제거
- 크레아티닌 절대치 기반 '신기능 고위험' 대신 eGFR(2021 CKD-EPI)와 UACR(URDACT) 추가
- 운동량은 vigorous/moderate 주간 분량으로 재구성
- 상관/회귀는 같은 subset에서 계산
- 결과 저장, 회귀표 저장, 그림 저장 포함

## 주의
- NHANES의 '정확한' 복합표본설계 분산추정은 Python만으로는 불편합니다.
- 본 코드는 WTDRD1을 사용한 가중 회귀(freq_weights)까지 반영한 실용형 버전입니다.
- 논문 제출용 최종분석은 R survey 패키지로 재현하는 것을 권장합니다.

## STEP 0. 패키지 설치 / 환경


In [ ]:
import os
import sys
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle', 'statsmodels', 'openpyxl'])
import re
import glob
import math
import shutil
import warnings
import subprocess
from pathlib import Path
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings('ignore')

# 한글 폰트
subprocess.run(['apt-get', '-qq', 'install', 'fonts-nanum'], check=False)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams.update({
    'axes.unicode_minus': False,
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PALETTE = ['#2D6A4F', '#52B788', '#95D5B2', '#D8F3DC', '#E76F51', '#F4A261', '#264653', '#457B9D']
GREEN = '#2D6A4F'
ACCENT = '#E76F51'
BLUE = '#457B9D'



## STEP 1. Kaggle 인증 + 데이터 다운로드


In [ ]:
from google.colab import files
uploaded = files.upload()  # kaggle.json 업로드
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

subprocess.check_call(['kaggle', 'datasets', 'download', '-d', 'cdc/national-health-and-nutrition-examination-survey', '--unzip', '-q'])

csvs = sorted(glob.glob('*.csv'))
print('다운로드된 CSV:', csvs)



## STEP 2. 파일 탐색


In [ ]:

def find_first(candidates, file_list):
    file_list_lower = {f.lower(): f for f in file_list}

    # 1) 완전 일치 우선
    for cand in candidates:
        if cand.lower() in file_list_lower:
            return file_list_lower[cand.lower()]

    # 2) 부분 포함
    for cand in candidates:
        for f in file_list:
            if cand.lower() in f.lower():
                return f
    return None

FILE_MAP = {
    'demo': find_first(['demographic.csv', 'demo', 'DEMO_H'], csvs),
    'diet': find_first(['diet.csv', 'dr1tot', 'DR1TOT_H'], csvs),
    'exam': find_first(['examination.csv', 'bmx', 'BMX_H'], csvs),
    'paq':  find_first(['questionnaire.csv', 'paq', 'PAQ_H'], csvs),
    'lab':  find_first(['labs.csv', 'biopro', 'BIOPRO_H'], csvs),
}

print('\n[파일 매핑 결과]')
for k, v in FILE_MAP.items():
    print(f'{k:>5}: {v}')

missing_files = [k for k, v in FILE_MAP.items() if v is None]
if missing_files:
    raise FileNotFoundError(f'다음 파일을 찾지 못했습니다: {missing_files}')



## STEP 3. 파일 로드


In [ ]:

demo = pd.read_csv(FILE_MAP['demo'])
diet = pd.read_csv(FILE_MAP['diet'])
exam = pd.read_csv(FILE_MAP['exam'])
paq  = pd.read_csv(FILE_MAP['paq'])
lab  = pd.read_csv(FILE_MAP['lab'])

print('\n[원본 크기]')
for name, df0 in [('demo', demo), ('diet', diet), ('exam', exam), ('paq', paq), ('lab', lab)]:
    print(f'{name}: {df0.shape}')



## STEP 4. 필요한 변수만 선택


In [ ]:

def keep_existing(df, cols):
    return df[[c for c in cols if c in df.columns]].copy()

# Demographics
DEMO_COLS = [
    'SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'SDMVPSU', 'SDMVSTRA',
    'WTMEC2YR', 'WTINT2YR', 'RIDEXPRG'
]

# Diet day 1
DIET_COLS = [
    'SEQN', 'DR1TPROT', 'DR1TKCAL', 'WTDRD1', 'WTDR2D', 'DR1DRSTZ',
    'DRQSDIET', 'DRQSDT10'
]

# Body measures
EXAM_COLS = ['SEQN', 'BMXWT', 'BMXHT', 'BMXBMI']

# Physical activity
PAQ_COLS = [
    'SEQN',
    'PAQ650', 'PAQ655', 'PAD660',  # vigorous work? / vigorous leisure depends on dataset version; keep as documented names
    'PAQ665', 'PAQ670', 'PAD675',
    'PAQ706', 'PAD680'
]

# Labs
LAB_COLS = ['SEQN', 'LBXSCR', 'LBXSBU', 'URDACT', 'URXUMA', 'URXUCR']

demo_use = keep_existing(demo, DEMO_COLS)
diet_use = keep_existing(diet, DIET_COLS)
exam_use = keep_existing(exam, EXAM_COLS)
paq_use  = keep_existing(paq, PAQ_COLS)
lab_use  = keep_existing(lab, LAB_COLS)

# =========================================================
# STEP 5. 병합
# - DEMO를 기준으로 left merge 후, 분석 시 필요한 변수 결측 제거
# =========================================================

df = demo_use.merge(diet_use, on='SEQN', how='left') \
             .merge(exam_use, on='SEQN', how='left') \
             .merge(paq_use,  on='SEQN', how='left') \
             .merge(lab_use,  on='SEQN', how='left')

print(f'\n병합 결과: {df.shape[0]:,}명 × {df.shape[1]}개 변수')



## STEP 6. 특수 결측코드 정리


In [ ]:

def clean_special_missing(series):
    s = pd.to_numeric(series, errors='coerce')
    # NHANES에서 자주 쓰는 특수코드들: 7/9 반복형, 7777/9999 등
    specials = {7, 77, 777, 7777, 77777, 9, 99, 999, 9999, 99999}
    s = s.replace(list(specials), np.nan)
    return s

num_like_cols = [c for c in df.columns if c != 'SEQN']
for c in num_like_cols:
    df[c] = clean_special_missing(df[c])



## STEP 7. 변수명 표준화


In [ ]:

rename_map = {
    'RIDAGEYR': 'age',
    'RIAGENDR': 'sex',
    'RIDRETH3': 'race_ethnicity',
    'SDMVPSU': 'psu',
    'SDMVSTRA': 'strata',
    'WTMEC2YR': 'wt_mec',
    'WTINT2YR': 'wt_int',
    'WTDRD1': 'wt_diet_day1',
    'WTDR2D': 'wt_diet_2day',
    'RIDEXPRG': 'pregnant',
    'DR1TPROT': 'protein_g_day',
    'DR1TKCAL': 'energy_kcal',
    'DR1DRSTZ': 'diet_status_day1',
    'DRQSDIET': 'on_special_diet',
    'DRQSDT10': 'renal_diet_flag',
    'BMXWT': 'weight_kg',
    'BMXHT': 'height_cm',
    'BMXBMI': 'bmi',
    'PAQ650': 'vigorous_flag',
    'PAQ655': 'vigorous_days',
    'PAD660': 'vigorous_min_day',
    'PAQ665': 'moderate_flag',
    'PAQ670': 'moderate_days',
    'PAD675': 'moderate_min_day',
    'PAQ706': 'days_active_60min',
    'PAD680': 'sedentary_min_day',
    'LBXSCR': 'scr_mg_dl',
    'LBXSBU': 'bun_mg_dl',
    'URDACT': 'uacr_mg_g',
    'URXUMA': 'urine_albumin_mg_l',
    'URXUCR': 'urine_creatinine_mg_dl',
}

df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})



## STEP 8. 대상자 선정


In [ ]:

# 성인만
if 'age' in df.columns:
    df = df[df['age'] >= 19].copy()

# 임신 제외 (값 1=Yes 인 경우 제외; 값 체계가 다르면 이 줄 삭제 가능)
if 'pregnant' in df.columns:
    df = df[(df['pregnant'].isna()) | (df['pregnant'] != 1)].copy()

# 식이 day1 기본 품질: 단백질, 에너지, day1 weight, creatinine 중 핵심 변수 보유자
required = ['protein_g_day', 'energy_kcal', 'scr_mg_dl', 'age', 'sex', 'weight_kg', 'bmi', 'wt_diet_day1']
required = [c for c in required if c in df.columns]

before = len(df)
df = df.dropna(subset=required).copy()
print(f'\n성인/핵심변수 필터: {before:,}명 → {len(df):,}명')

# 비현실적 값 최소 필터
if 'energy_kcal' in df.columns:
    df = df[(df['energy_kcal'] > 0) & (df['energy_kcal'] < 10000)].copy()
if 'protein_g_day' in df.columns:
    df = df[(df['protein_g_day'] > 0) & (df['protein_g_day'] < 500)].copy()
if 'weight_kg' in df.columns:
    df = df[(df['weight_kg'] > 20) & (df['weight_kg'] < 300)].copy()
if 'height_cm' in df.columns:
    df = df[(df['height_cm'].isna()) | ((df['height_cm'] > 100) & (df['height_cm'] < 250))].copy()
if 'scr_mg_dl' in df.columns:
    df = df[(df['scr_mg_dl'] > 0.2) & (df['scr_mg_dl'] < 15)].copy()

print(f'값 범위 필터 후: {len(df):,}명')



## STEP 9. 파생변수 생성


In [ ]:

def egfr_ckdepi_2021(scr_mg_dl, age, sex_code):
    """
    2021 CKD-EPI creatinine equation
    sex_code: NHANES 기준 1=male, 2=female
    """
    scr = pd.to_numeric(scr_mg_dl, errors='coerce')
    age = pd.to_numeric(age, errors='coerce')
    sex = pd.to_numeric(sex_code, errors='coerce')

    female = (sex == 2)
    kappa = np.where(female, 0.7, 0.9)
    alpha = np.where(female, -0.241, -0.302)
    female_mult = np.where(female, 1.012, 1.0)

    scr_k = scr / kappa
    egfr = 142 * np.minimum(scr_k, 1) ** alpha * np.maximum(scr_k, 1) ** (-1.200) * (0.9938 ** age) * female_mult
    return egfr

# 식이 노출
df['protein_gkg_day'] = df['protein_g_day'] / df['weight_kg']
df['protein_pct_energy'] = (df['protein_g_day'] * 4 / df['energy_kcal']) * 100

# 운동량
for c in ['vigorous_days', 'vigorous_min_day', 'moderate_days', 'moderate_min_day']:
    if c in df.columns:
        df[c] = df[c].fillna(0)

# flag가 2(아니오)면 해당 값 0 처리
if 'vigorous_flag' in df.columns:
    df.loc[df['vigorous_flag'] == 2, ['vigorous_days', 'vigorous_min_day']] = 0
if 'moderate_flag' in df.columns:
    df.loc[df['moderate_flag'] == 2, ['moderate_days', 'moderate_min_day']] = 0

df['vigorous_weekly_min'] = df.get('vigorous_days', 0) * df.get('vigorous_min_day', 0)
df['moderate_weekly_min'] = df.get('moderate_days', 0) * df.get('moderate_min_day', 0)
df['mvpa_weekly_equiv'] = df['moderate_weekly_min'] + 2 * df['vigorous_weekly_min']

# 활동군
# inactive: 0, insufficient: 0~149, active: >=150 등가분
bins = [-0.1, 0, 149.999, np.inf]
labels = ['inactive', 'insufficient', 'active']
df['activity_group'] = pd.cut(df['mvpa_weekly_equiv'], bins=bins, labels=labels)

# 신장지표
df['egfr_2021'] = egfr_ckdepi_2021(df['scr_mg_dl'], df['age'], df['sex'])
df['reduced_egfr'] = (df['egfr_2021'] < 60).astype(int)

df['albuminuria'] = np.where(df['uacr_mg_g'].notna(), (df['uacr_mg_g'] >= 30).astype(int), np.nan)
df['kidney_marker_risk'] = np.where(
    (df['reduced_egfr'] == 1) | (df['albuminuria'] == 1),
    '고위험', '정상'
)

# UACR 범주
def uacr_cat(x):
    if pd.isna(x):
        return np.nan
    if x < 30:
        return 'A1(<30)'
    elif x < 300:
        return 'A2(30-299)'
    return 'A3(≥300)'

df['uacr_category'] = df['uacr_mg_g'].apply(uacr_cat)

# 단백질 그룹: g/day, g/kg/day 둘 다 생성
df['protein_group_g'] = pd.qcut(df['protein_g_day'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
df['protein_group_gkg'] = pd.qcut(df['protein_gkg_day'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
df['high_protein_1_2gkg'] = (df['protein_gkg_day'] >= 1.2).astype(int)

# 로그 UACR (연속형 분석용)
df['log_uacr'] = np.log(df['uacr_mg_g'].clip(lower=0.1))

# 시각화용 윈저라이즈 버전(분석 원자료는 보존)
def winsorize(s, q=0.01):
    lo, hi = s.quantile(q), s.quantile(1-q)
    return s.clip(lo, hi)

for src, dst in [
    ('protein_g_day', 'protein_g_day_w'),
    ('protein_gkg_day', 'protein_gkg_day_w'),
    ('scr_mg_dl', 'scr_mg_dl_w'),
    ('bun_mg_dl', 'bun_mg_dl_w'),
    ('egfr_2021', 'egfr_2021_w'),
    ('uacr_mg_g', 'uacr_mg_g_w'),
    ('bmi', 'bmi_w'),
]:
    if src in df.columns:
        df[dst] = winsorize(df[src].dropna()).reindex(df.index)

print('\n[핵심 파생변수 요약]')
print(df[['protein_g_day', 'protein_gkg_day', 'scr_mg_dl', 'egfr_2021', 'uacr_mg_g', 'mvpa_weekly_equiv']].describe().round(2))



## STEP 10. 기술통계 테이블


In [ ]:

def weighted_mean(x, w):
    x = pd.to_numeric(x, errors='coerce')
    w = pd.to_numeric(w, errors='coerce')
    mask = x.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(x[mask], weights=w[mask])

baseline_vars = ['age', 'bmi', 'protein_g_day', 'protein_gkg_day', 'energy_kcal', 'scr_mg_dl', 'egfr_2021', 'uacr_mg_g', 'mvpa_weekly_equiv']
baseline_rows = []

for grp in sorted(df['protein_group_gkg'].dropna().unique()):
    sub = df[df['protein_group_gkg'] == grp].copy()
    row = {'protein_group_gkg': grp, 'n': len(sub)}
    for v in baseline_vars:
        if v in sub.columns:
            row[f'{v}_mean_w'] = weighted_mean(sub[v], sub['wt_diet_day1'])
    if 'kidney_marker_risk' in sub.columns:
        risk = sub['kidney_marker_risk'].eq('고위험')
        row['kidney_marker_risk_pct_w'] = weighted_mean(risk.astype(float), sub['wt_diet_day1']) * 100
    baseline_rows.append(row)

baseline_table = pd.DataFrame(baseline_rows)
print('\n[기술통계 테이블: protein_group_gkg 기준]')
print(baseline_table.round(2))



## STEP 11. 회귀모형


In [ ]:
# 주의: freq_weights는 NHANES 복합표본분산을 완전히 대체하지 못함.
# 실용형 Python 분석용. 논문 최종치는 R survey 권장.
# =========================================================

analysis_cols = [
    'protein_gkg_day', 'protein_g_day', 'protein_pct_energy',
    'age', 'sex', 'bmi', 'energy_kcal', 'mvpa_weekly_equiv',
    'scr_mg_dl', 'egfr_2021', 'log_uacr', 'uacr_mg_g', 'albuminuria',
    'reduced_egfr', 'wt_diet_day1', 'psu', 'strata', 'race_ethnicity',
    'protein_group_gkg', 'activity_group'
]
analysis_cols = [c for c in analysis_cols if c in df.columns]
ana = df[analysis_cols].copy()

# 범주형 변환
for c in ['sex', 'race_ethnicity', 'activity_group', 'protein_group_gkg']:
    if c in ana.columns:
        ana[c] = ana[c].astype('category')

# 1) 선형회귀: creatinine
lin1 = ana.dropna(subset=['scr_mg_dl', 'protein_gkg_day', 'age', 'sex', 'bmi', 'energy_kcal', 'mvpa_weekly_equiv', 'wt_diet_day1']).copy()
model_scr = smf.wls(
    'scr_mg_dl ~ protein_gkg_day + age + C(sex) + bmi + energy_kcal + mvpa_weekly_equiv',
    data=lin1,
    weights=lin1['wt_diet_day1']
).fit(cov_type='HC1')

# 2) 선형회귀: eGFR
lin2 = ana.dropna(subset=['egfr_2021', 'protein_gkg_day', 'age', 'sex', 'bmi', 'energy_kcal', 'mvpa_weekly_equiv', 'wt_diet_day1']).copy()
model_egfr = smf.wls(
    'egfr_2021 ~ protein_gkg_day + age + C(sex) + bmi + energy_kcal + mvpa_weekly_equiv',
    data=lin2,
    weights=lin2['wt_diet_day1']
).fit(cov_type='HC1')

# 3) 선형회귀: log UACR
lin3 = ana.dropna(subset=['log_uacr', 'protein_gkg_day', 'age', 'sex', 'bmi', 'energy_kcal', 'mvpa_weekly_equiv', 'wt_diet_day1']).copy()
if len(lin3) > 100:
    model_uacr = smf.wls(
        'log_uacr ~ protein_gkg_day + age + C(sex) + bmi + energy_kcal + mvpa_weekly_equiv',
        data=lin3,
        weights=lin3['wt_diet_day1']
    ).fit(cov_type='HC1')
else:
    model_uacr = None

# 4) 로지스틱: reduced eGFR
log1 = ana.dropna(subset=['reduced_egfr', 'protein_gkg_day', 'age', 'sex', 'bmi', 'energy_kcal', 'mvpa_weekly_equiv', 'wt_diet_day1']).copy()
model_reduced_egfr = smf.glm(
    'reduced_egfr ~ protein_gkg_day + age + C(sex) + bmi + energy_kcal + mvpa_weekly_equiv',
    data=log1,
    family=sm.families.Binomial(),
    freq_weights=log1['wt_diet_day1']
).fit(cov_type='HC1')

# 5) 로지스틱: albuminuria
log2 = ana.dropna(subset=['albuminuria', 'protein_gkg_day', 'age', 'sex', 'bmi', 'energy_kcal', 'mvpa_weekly_equiv', 'wt_diet_day1']).copy()
model_albuminuria = smf.glm(
    'albuminuria ~ protein_gkg_day + age + C(sex) + bmi + energy_kcal + mvpa_weekly_equiv',
    data=log2,
    family=sm.families.Binomial(),
    freq_weights=log2['wt_diet_day1']
).fit(cov_type='HC1')

# 6) interaction model: scr ~ protein * activity_group
int1 = ana.dropna(subset=['scr_mg_dl', 'protein_gkg_day', 'activity_group', 'age', 'sex', 'bmi', 'energy_kcal', 'wt_diet_day1']).copy()
model_interaction_scr = smf.wls(
    'scr_mg_dl ~ protein_gkg_day * C(activity_group) + age + C(sex) + bmi + energy_kcal',
    data=int1,
    weights=int1['wt_diet_day1']
).fit(cov_type='HC1')

print('\n[회귀 요약: creatinine]')
print(model_scr.summary())
print('\n[회귀 요약: eGFR]')
print(model_egfr.summary())
print('\n[회귀 요약: reduced eGFR - logistic]')
print(model_reduced_egfr.summary())
print('\n[회귀 요약: albuminuria - logistic]')
print(model_albuminuria.summary())
print('\n[회귀 요약: interaction(scr)]')
print(model_interaction_scr.summary())



## STEP 12. 회귀 결과 정리 함수


In [ ]:

def tidy_model(model, exponentiate=False, keep_terms=None):
    res = pd.DataFrame({
        'term': model.params.index,
        'coef': model.params.values,
        'se': model.bse.values,
        'p': model.pvalues.values,
    })
    ci = model.conf_int()
    res['ci_low'] = ci[0].values
    res['ci_high'] = ci[1].values
    if exponentiate:
        for c in ['coef', 'ci_low', 'ci_high']:
            res[c] = np.exp(res[c])
    if keep_terms is not None:
        res = res[res['term'].isin(keep_terms)].copy()
    return res

reg_scr = tidy_model(model_scr)
reg_egfr = tidy_model(model_egfr)
reg_uacr = tidy_model(model_uacr) if model_uacr is not None else pd.DataFrame()
reg_reduced_egfr = tidy_model(model_reduced_egfr, exponentiate=True)
reg_albuminuria = tidy_model(model_albuminuria, exponentiate=True)
reg_inter_scr = tidy_model(model_interaction_scr)



## STEP 13. 시각화


In [ ]:

# Figure 1. 단백질 g/kg/day 분포
plt.figure(figsize=(8, 5))
sns.histplot(df['protein_gkg_day_w'].dropna(), bins=35, color=GREEN, kde=True)
plt.axvline(df['protein_gkg_day'].median(), color=ACCENT, linestyle='--', label=f'중앙값 {df["protein_gkg_day"].median():.2f} g/kg/day')
plt.xlabel('체중보정 단백질 섭취량 (g/kg/day)')
plt.ylabel('인원 수')
plt.title('체중보정 단백질 섭취량 분포')
plt.legend()
plt.tight_layout()
plt.savefig('fig1_protein_gkg_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 2. 단백질 사분위별 creatinine / eGFR
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = ['Q1', 'Q2', 'Q3', 'Q4']

sns.boxplot(data=df, x='protein_group_gkg', y='scr_mg_dl_w', order=order, ax=axes[0], color='#F4A261')
axes[0].set_xlabel('단백질 사분위 (g/kg/day)')
axes[0].set_ylabel('Serum creatinine (mg/dL)')
axes[0].set_title('단백질 사분위별 혈청 크레아티닌')

sns.boxplot(data=df, x='protein_group_gkg', y='egfr_2021_w', order=order, ax=axes[1], color='#95D5B2')
axes[1].set_xlabel('단백질 사분위 (g/kg/day)')
axes[1].set_ylabel('eGFR (mL/min/1.73㎡)')
axes[1].set_title('단백질 사분위별 eGFR')

plt.tight_layout()
plt.savefig('fig2_protein_quartile_creatinine_egfr.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 3. 단백질 g/kg/day vs creatinine / eGFR
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sub1 = df[['protein_gkg_day_w', 'scr_mg_dl_w']].dropna()
sub2 = df[['protein_gkg_day_w', 'egfr_2021_w']].dropna()

sns.regplot(data=sub1, x='protein_gkg_day_w', y='scr_mg_dl_w', scatter_kws={'s': 12, 'alpha': 0.25}, line_kws={'color': '#264653'}, ax=axes[0])
axes[0].set_xlabel('단백질 섭취량 (g/kg/day)')
axes[0].set_ylabel('Serum creatinine (mg/dL)')
axes[0].set_title('단백질 vs 크레아티닌')

sns.regplot(data=sub2, x='protein_gkg_day_w', y='egfr_2021_w', scatter_kws={'s': 12, 'alpha': 0.25}, line_kws={'color': '#264653'}, ax=axes[1])
axes[1].set_xlabel('단백질 섭취량 (g/kg/day)')
axes[1].set_ylabel('eGFR (mL/min/1.73㎡)')
axes[1].set_title('단백질 vs eGFR')

plt.tight_layout()
plt.savefig('fig3_scatter_protein_creatinine_egfr.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 4. activity x protein heatmap (고위험 비율)
heat_df = df.dropna(subset=['protein_group_gkg', 'activity_group', 'kidney_marker_risk']).copy()
heat_pivot = heat_df.groupby(['protein_group_gkg', 'activity_group'])['kidney_marker_risk'] \
                   .apply(lambda x: (x == '고위험').mean() * 100) \
                   .unstack()

plt.figure(figsize=(7, 5))
sns.heatmap(heat_pivot, annot=True, fmt='.1f', cmap=sns.light_palette(ACCENT, as_cmap=True), linewidths=1, linecolor='white')
plt.xlabel('운동군')
plt.ylabel('단백질 사분위 (g/kg/day)')
plt.title('단백질 × 운동군별 신장 위험표지 고위험 비율 (%)')
plt.tight_layout()
plt.savefig('fig4_heatmap_protein_activity_kidneyrisk.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 5. interaction plot (단순 예측선)
pred_grid = pd.DataFrame({
    'protein_gkg_day': np.linspace(df['protein_gkg_day'].quantile(0.01), df['protein_gkg_day'].quantile(0.99), 100)
})

pred_frames = []
mean_age = df['age'].mean()
mean_bmi = df['bmi'].mean()
mean_energy = df['energy_kcal'].mean()
ref_sex = df['sex'].mode().iloc[0]

for grp in ['inactive', 'insufficient', 'active']:
    tmp = pred_grid.copy()
    tmp['activity_group'] = grp
    tmp['age'] = mean_age
    tmp['bmi'] = mean_bmi
    tmp['energy_kcal'] = mean_energy
    tmp['sex'] = ref_sex
    tmp['pred_scr'] = model_interaction_scr.predict(tmp)
    pred_frames.append(tmp)

pred_plot = pd.concat(pred_frames, ignore_index=True)

plt.figure(figsize=(8, 5))
for grp, color in zip(['inactive', 'insufficient', 'active'], [ACCENT, BLUE, GREEN]):
    ss = pred_plot[pred_plot['activity_group'] == grp]
    plt.plot(ss['protein_gkg_day'], ss['pred_scr'], label=grp, color=color, lw=2)
plt.xlabel('단백질 섭취량 (g/kg/day)')
plt.ylabel('예측 Serum creatinine (mg/dL)')
plt.title('운동군에 따른 단백질-크레아티닌 연관성')
plt.legend(title='운동군')
plt.tight_layout()
plt.savefig('fig5_interaction_protein_activity_scr.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 6. UACR 범주 분포
uacr_ct = pd.crosstab(df['protein_group_gkg'], df['uacr_category'], normalize='index') * 100
uacr_ct = uacr_ct[[c for c in ['A1(<30)', 'A2(30-299)', 'A3(≥300)'] if c in uacr_ct.columns]]
uacr_ct.plot(kind='bar', stacked=True, figsize=(8, 5), color=['#95D5B2', '#F4A261', '#E76F51'])
plt.xlabel('단백질 사분위 (g/kg/day)')
plt.ylabel('비율 (%)')
plt.title('단백질 사분위별 UACR 범주 분포')
plt.legend(title='UACR')
plt.tight_layout()
plt.savefig('fig6_stacked_uacr_category.png', dpi=150, bbox_inches='tight')
plt.show()



## STEP 14. 상관분석 (같은 subset에서 계산)


In [ ]:

def pearson_complete_case(data, x, y):
    sub = data[[x, y]].dropna().copy()
    if len(sub) < 3:
        return np.nan, np.nan, len(sub)
    r, p = stats.pearsonr(sub[x], sub[y])
    return r, p, len(sub)

for x, y in [('protein_g_day', 'scr_mg_dl'), ('protein_gkg_day', 'scr_mg_dl'), ('protein_gkg_day', 'egfr_2021'), ('protein_gkg_day', 'log_uacr')]:
    r, p, n = pearson_complete_case(df, x, y)
    print(f'상관: {x} vs {y} | n={n:,}, r={r:.4f}, p={p:.4g}')



## STEP 15. 저장


In [ ]:

out_dir = Path('nhanes_outputs')
out_dir.mkdir(exist_ok=True)

# 데이터 저장
save_df = df.copy()
save_df.to_csv(out_dir / 'analysis_dataset.csv', index=False, encoding='utf-8-sig')
baseline_table.to_csv(out_dir / 'table1_baseline_by_protein_group.csv', index=False, encoding='utf-8-sig')
reg_scr.to_csv(out_dir / 'reg_linear_scr.csv', index=False, encoding='utf-8-sig')
reg_egfr.to_csv(out_dir / 'reg_linear_egfr.csv', index=False, encoding='utf-8-sig')
if not reg_uacr.empty:
    reg_uacr.to_csv(out_dir / 'reg_linear_loguacr.csv', index=False, encoding='utf-8-sig')
reg_reduced_egfr.to_csv(out_dir / 'reg_logistic_reduced_egfr_or.csv', index=False, encoding='utf-8-sig')
reg_albuminuria.to_csv(out_dir / 'reg_logistic_albuminuria_or.csv', index=False, encoding='utf-8-sig')
reg_inter_scr.to_csv(out_dir / 'reg_interaction_scr.csv', index=False, encoding='utf-8-sig')

# 그림 이동
for fig in glob.glob('fig*.png'):
    shutil.move(fig, out_dir / Path(fig).name)

print('\n저장 완료 파일 목록')
for p in sorted(out_dir.glob('*')):
    print(f'- {p.name} ({p.stat().st_size/1024:.1f} KB)')

print('\n✅ 수정 분석 완료')
print('다음 추천: 이 결과를 바탕으로 R survey 패키지로 최종 복합표본분석 재현')
